In [3]:
import numpy as np 
import obspy
import matplotlib.pyplot as plt
import os,glob

Create an inventory of stations located in Greenland.

In [ ]:
from obspy.clients.fdsn import Client
# from obspy import UTCDateTime

client = Client("IRIS")
inv=client.get_stations(minlatitude=59.50, maxlatitude=83.80, 
                        minlongitude=-73.50, maxlongitude=-11.00,
                        # starttime=time1,endtime=time2,
                        network='*',channel='?H?',
                        level='channel')


# print(inv)

Inventory created at 2026-06-27T18:55:17.427800Z
	Created by: IRIS WEB SERVICE: fdsnws-station | version: 1.1.52
		    http://service.iris.edu/fdsnws/station/1/query?network=%2A&channel=...
	Sending institution: IRIS-DMC (IRIS-DMC)
	Contains:
		Networks (47):
			14, 3A, 4F, 5Q, 6H, 7E, 7J, 8F, 9C, 9D, CN, DK, DT, DW, G, GE, GG, 
			II, IU, KP, PO, X3, X5, XD, XE (2x), XF, XH, XV, XW, XY, Y6, YA, YE
			YF, YM, YQ, YT, YV, YY, Z7, ZJ, ZK, ZN (2x), ZO, ZS
		Stations (755):
			14.ISR0 (Isunnguata Sermia Terminus)
			3A.DJPA (DjÃºpavatn, Iceland)
			3A.KILR (North of Keilir, Iceland)
			4F.BRUN (Utbruni)
			4F.DDAL (Dyngjufjalladalur)
			4F.DYNG (Dyngja)
			4F.FJAL (Fjallsendi)
			4F.FJAS (Fjalsendi 2)
			4F.FLAT (Flatadyngja)
			4F.FLUR (Flaedur)
			4F.FREF (Fremstafell)
			4F.HELI (Herdubreidarlindir)
			4F.HERD (Herdubreid)
			4F.HETO (Herdubreidartogl)
			4F.HOTT (Hottur)
			4F.HRIM (Hrimalda)
			4F.HRUR (Hrutur 2)
			4F.HRUT (Hrutur)
			4F.JOAF (Jokulsa a fjollum 2)
			4F.KOLL (Kollott

In [ ]:
from shapely.geometry import Point, Polygon

# Greenland polygon (Longitude, Latitude)
greenland_coords = [
    (-44.0, 59.5),   # 1. S Tip (Cape Farewell)
    (-55.0, 60.0),   # 2. SW Coast (Labrador Sea)
    (-62.0, 70.0),   # 3. MidW (Baffin Bay)
    (-74.0, 77.0),   # 4. W Point (Thule area)
    (-61.0, 82.2),   # 5. NW Corner (Nares Strait border)
    (-34.0, 83.8),   # 6. N Tip (Cape Morris Jesup)
    (-11.0, 81.5),   # 7. NE Corner (Nordostrundingen)
    (-20.0, 75.0),   # 8. Greenland Sea 
    (-30.0, 69.0),   # 9. Denmark Strait Upper (N Iceland)
    (-42.0, 62.0),   # 10. Denmark Strait Lower (W Iceland)
    (-44.0, 59.5)    # Close loop
]

greenland_polygon = Polygon(greenland_coords)

inventory = inv.copy() # Copy the inventory to filter, inv_filtered

not_greenland_networks = []
for network in inventory:

    greenland_stations = []
    for station in network:
        coord = Point(station.longitude, station.latitude)
        if greenland_polygon.contains(coord):
            greenland_stations.append(station)
    network.stations = greenland_stations
    
    if len(greenland_stations) == 0:
        not_greenland_networks.append(network)
for n in not_greenland_networks: 
    inventory.networks.remove(n)    

print(inventory) 

Inventory created at 2026-06-27T18:55:17.427800Z
	Created by: IRIS WEB SERVICE: fdsnws-station | version: 1.1.52
		    http://service.iris.edu/fdsnws/station/1/query?network=%2A&channel=...
	Sending institution: IRIS-DMC (IRIS-DMC)
	Contains:
		Networks (23):
			14, 5Q, 6H, 7E, 9C, 9D, DK, DW, G, GE, GG, IU, KP, X3, XF, XW, Y6, 
			YE, YF, YM, YV, ZN, ZS
		Stations (130):
			14.ISR0 (Isunnguata Sermia Terminus)
			5Q.SE47 (Transect site, southern most, Ilulissat, Greenland)
			5Q.SE50 (Transect site, 2rd from south, Ilulissat, Greenland)
			5Q.SE53 (Transect site, 3rd from south, Ilulissat, Greenland)
			5Q.SE57 (Transect site, 4rd from south, Ilulissat, Greenland)
			5Q.SE60 (Transect site, 5rd from south, Ilulissat, Greenland)
			5Q.SE63 (Transect site, 6th from south, Ilulissat, Greenland)
			5Q.SEHC (Seismic site at MoVE high camp, Ilulissat, Greenland)
			5Q.SELC (Seismic site at MoVE low camp, Ilulissat, Greenland)
			6H.GL11 (GL11)
			6H.GL12 (GL12)
			6H.GL13 (GL13)
			6H.GL14 